# 08 · Error analysis, Grad-CAM and the "not sure" threshold (plan stage 5)

Works on predictions saved earlier, so the test split is **not** evaluated again (D-015):
`val_scores.npz` from training and `test_scores.npz` from the one-time test evaluation of the final model.

1. **Metrics (5.1):** row-normalised 38 × 38 confusion matrix, per-class precision / recall / F1 with counts.
2. **Comparison (5.2):** `reports/results.md`, generated from `reports/results.csv`.
3. **Error analysis (5.3):** the 5 most confused class pairs, the 5 weakest classes, and the full-test vs
   leaf-verified gap split into its two causes (missing classes vs residual leakage).
4. **Failure gallery (5.4):** 6 misclassified test images with Grad-CAM.
5. **Grad-CAM check (D-020):** how much of the heat map lies on the leaf, measured with the leaf masks of the
   dataset's *segmented* variant on 15 test images per class.
6. **"Not sure" threshold (D-019):** the smallest confidence threshold whose answered **validation** images are at
   least 99.5% correct; then applied unchanged to test.
7. The near-identical train/test pairs that the duplicate search removed (D-006 figure).

**No GPU needed** (a CPU runtime is fine); about 5–10 minutes.

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly (D-009)
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline
!nvidia-smi -L || echo "no GPU"
!python -m plantdx.env

## 2 · Images (colour for the model, segmented for the leaf masks)

In [ ]:
from plantdx.data.images import download_archive, extract_variant

DATA_ROOT = "/content/data"
archive = download_archive()
for variant in ("color", "segmented"):
    print(variant, "new files extracted:", extract_variant(archive, DATA_ROOT, variant))

## 3 · Run the analysis

In [ ]:
from plantdx.analyze import run

summary = run(DATA_ROOT)

## 4 · Figures ← **look at these**

In [ ]:
from pathlib import Path
from IPython.display import Image as IPImage, Markdown, display
from plantdx.paths import artifacts_dir

OUT = artifacts_dir() / "stage5"
FIGS = OUT / "reports" / "figures"
for name in ["confusion_matrix.png", "failures.jpg", "gradcam_examples.jpg", "gradcam_on_leaf.png",
             "coverage_not_sure.png", "dup_leak_examples.jpg"]:
    if (FIGS / name).exists():
        print(name)
        display(IPImage(str(FIGS / name)))

## 5 · results.md

In [ ]:
display(Markdown((OUT / "reports" / "results.md").read_text()))

## 6 · Download

`stage5_analysis.zip` holds `reports/results.md`, `reports/analysis.json`, `reports/analysis/*.csv` and the new
figures. Send it in the chat.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/stage5_analysis", "zip", root_dir=OUT, base_dir="reports")
files.download(zip_path)